In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
billing_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.billing")

patients_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.patients")

treatments_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.treatments")

In [0]:
display(billing_df)

bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,ingestion_timestamp,source_file,load_date
B001,P034,T001,2023-08-09,3941.97,Insurance,Pending,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B002,P032,T002,2023-06-09,4158.44,Insurance,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B003,P048,T003,2023-06-28,3731.55,Insurance,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B004,P025,T004,2023-09-01,4799.86,Insurance,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B005,P040,T005,2023-07-06,582.05,Credit Card,Pending,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B006,P045,T006,2023-06-19,1381.0,Insurance,Pending,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B007,P001,T007,2023-04-09,534.03,Cash,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B008,P016,T008,2023-05-24,3413.64,Cash,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02
B010,P005,T010,2023-01-13,1595.67,Cash,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02


In [0]:
billing_df = billing_df.dropDuplicates(["bill_id"])

In [0]:
billing_df = billing_df.join(
    patients_df.select("patient_id"),
    on="patient_id",
    how="inner"
)

In [0]:
billing_df = billing_df.join(
    treatments_df.select("treatment_id"),
    on="treatment_id",
    how="inner"
)

In [0]:
billing_df = (

    billing_df

    .withColumn(
        "payment_method",
        initcap(trim(col("payment_method")))
    )

    .withColumn(
        "payment_status",
        initcap(trim(col("payment_status")))
    )

)

In [0]:
billing_df = billing_df.filter(
    col("amount") > 0
)

In [0]:
billing_df = (

    billing_df

    .withColumn(
        "paid_flag",
        when(col("payment_status") == "Paid", 1).otherwise(0)
    )

    .withColumn(
        "pending_flag",
        when(col("payment_status") == "Pending", 1).otherwise(0)
    )

    .withColumn(
        "failed_flag",
        when(col("payment_status") == "Failed", 1).otherwise(0)
    )

)

In [0]:
billing_df = billing_df.withColumn(

    "successful_payment",

    when(col("payment_status") == "Paid", col("amount"))

    .otherwise(0)

)

In [0]:
billing_df = (

    billing_df

    .withColumn(
        "bill_year",
        year(col("bill_date"))
    )

    .withColumn(
        "bill_month",
        month(col("bill_date"))
    )

)

In [0]:
billing_df = billing_df.withColumn(

    "silver_load_timestamp",

    current_timestamp()

)

In [0]:
(
    billing_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.billing")
)

In [0]:
display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.billing")
)

treatment_id,patient_id,bill_id,bill_date,amount,payment_method,payment_status,ingestion_timestamp,source_file,load_date,paid_flag,pending_flag,failed_flag,successful_payment,bill_year,bill_month,silver_load_timestamp
T046,P019,B046,2023-12-20,1526.36,Cash,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,1,0,0,1526.36,2023,12,2026-07-02T07:36:42.636Z
T073,P040,B073,2023-12-24,2259.08,Credit Card,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,12,2026-07-02T07:36:42.636Z
T100,P029,B100,2023-03-02,1551.7,Credit Card,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,3,2026-07-02T07:36:42.636Z
T102,P025,B102,2023-10-25,4460.36,Credit Card,Pending,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,1,0,0.0,2023,10,2026-07-02T07:36:42.636Z
T150,P047,B150,2023-08-16,2286.42,Credit Card,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,1,0,0,2286.42,2023,8,2026-07-02T07:36:42.636Z
T165,P031,B165,2023-04-04,4126.66,Cash,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,4,2026-07-02T07:36:42.636Z
T168,P023,B168,2023-09-29,864.14,Credit Card,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,9,2026-07-02T07:36:42.636Z
T198,P022,B198,2023-05-15,3383.72,Cash,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,5,2026-07-02T07:36:42.636Z
T199,P017,B199,2023-05-01,1472.17,Credit Card,Paid,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,1,0,0,1472.17,2023,5,2026-07-02T07:36:42.636Z
T021,P028,B021,2023-04-24,2926.23,Insurance,Failed,2026-07-02T06:36:54.290Z,billing.csv,2026-07-02,0,0,1,0.0,2023,4,2026-07-02T07:36:42.636Z


In [0]:
for table in ["patients", "doctors", "appointments", "treatments", "billing"]:
    count = spark.table(f"workspace.silver.{table}").count()
    print(f"{table}: {count} records")

patients: 50 records
doctors: 10 records
appointments: 200 records
treatments: 200 records
billing: 200 records
